# Диаграммы персистентности для фазовых вложений временных рядов

На данном этапе для каждого временного ряда используются ранее найденные оптимальные параметры фазового вложения: лаг `tau` и размерность `dimension`.  
Для каждого ряда строится облако точек в фазовом пространстве, после чего по этому облаку вычисляются диаграммы персистентности.

Расчёт диаграмм выполняется в Julia с помощью библиотеки `Ripserer.jl`, а визуализация результатов выполняется в Python.

In [ ]:
import subprocess

packages = ["CSV", "DataFrames", "DelayEmbeddings", "Ripserer"]

for package in packages:
    print(f"Installing/checking {package}...")

    result = subprocess.run(
        [
            "julia",
            "--project=.",
            "-e",
            f'import Pkg; Pkg.add("{package}")'
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Ошибка при установке пакета {package}")

Installing/checking CSV...


In [ ]:
import subprocess

process = subprocess.Popen(
    ["julia", "--project=.", "persistence_diagrams.jl"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

Reading dataset: dataset_clean_31.csv
Reading best embedding parameters: uzal_cost_final_result.csv
Columns in dataset: 32
Rows in best parameters table: 75
Rows selected for persistence diagrams: 27
[1/27] Lab1_Hpol
в”Њ Warning: points not unique
в”” @ Ripserer C:\Users\margo\.julia\packages\Ripserer\f1W5D\src\filtrations\rips.jl:228
[2/27] Lab1_dev
в”Њ Warning: points not unique
в”” @ Ripserer C:\Users\margo\.julia\packages\Ripserer\f1W5D\src\filtrations\rips.jl:228
[3/27] Lab1_G3_T600
[4/27] Lab1_dPmg
в”Њ Warning: points not unique
в”” @ Ripserer C:\Users\margo\.julia\packages\Ripserer\f1W5D\src\filtrations\rips.jl:228
[5/27] Lab1_PposleNag
в”Њ Warning: points not unique
в”” @ Ripserer C:\Users\margo\.julia\packages\Ripserer\f1W5D\src\filtrations\rips.jl:228
[6/27] Lab1_TC_Ptgdg
в”Њ Warning: points not unique
в”” @ Ripserer C:\Users\margo\.julia\packages\Ripserer\f1W5D\src\filtrations\rips.jl:228
[7/27] Lab1_G2_FРЅ9
в”Њ Warning: points not unique
в”” @ Ripserer C:\Users\margo\.julia

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PERSISTENCE_PATH = Path("persistence_diagrams_31_columns.csv")
SKIPPED_PATH = Path("persistence_diagrams_31_skipped_columns.csv")

diagrams = pd.read_csv(PERSISTENCE_PATH)

diagrams.head()

,column,tau,dimension,homology_dimension,birth,death,persistence,is_infinite
0,Lab1_G3_Lm,32,3,0,0.0,0.000040,0.000040,False
1,Lab1_G3_Lm,32,3,0,0.0,0.000045,0.000045,False
2,Lab1_G3_Lm,32,3,0,0.0,0.000053,0.000053,False
3,Lab1_G3_Lm,32,3,0,0.0,0.000054,0.000054,False
4,Lab1_G3_Lm,32,3,0,0.0,0.000055,0.000055,False


In [ ]:
print("Количество точек диаграмм:", len(diagrams))
print("Количество рядов:", diagrams["column"].nunique())

diagrams.head()

Количество точек диаграмм: 119470
Количество рядов: 75


,column,tau,dimension,homology_dimension,birth,death,persistence,is_infinite
0,Lab1_G3_Lm,32,3,0,0.0,0.000040,0.000040,False
1,Lab1_G3_Lm,32,3,0,0.0,0.000045,0.000045,False
2,Lab1_G3_Lm,32,3,0,0.0,0.000053,0.000053,False
3,Lab1_G3_Lm,32,3,0,0.0,0.000054,0.000054,False
4,Lab1_G3_Lm,32,3,0,0.0,0.000055,0.000055,False


In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output


def plot_persistence_diagram_for_column(column_name):
    one = diagrams[diagrams["column"] == column_name].copy()

    # Для обычной диаграммы birth-death не отображаем бесконечные интервалы,
    # потому что death = Inf нельзя нормально положить на конечную ось.
    finite = one[
        np.isfinite(one["birth"]) &
        np.isfinite(one["death"])
    ].copy()

    if finite.empty:
        fig = go.Figure()
        fig.update_layout(
            title=f"Нет конечных точек диаграммы для ряда: {column_name}",
            width=800,
            height=650
        )
        fig.show()
        return

    max_value = max(finite["birth"].max(), finite["death"].max())
    min_value = min(finite["birth"].min(), finite["death"].min())

    padding = 0.05 * (max_value - min_value) if max_value > min_value else 1.0
    axis_min = min_value - padding
    axis_max = max_value + padding

    fig = go.Figure()

    for h_dim in sorted(finite["homology_dimension"].unique()):
        part = finite[finite["homology_dimension"] == h_dim]

        fig.add_trace(
            go.Scatter(
                x=part["birth"],
                y=part["death"],
                mode="markers",
                name=f"H{h_dim}",
                customdata=np.stack(
                    [
                        part["persistence"],
                        part["tau"],
                        part["dimension"]
                    ],
                    axis=-1
                ),
                hovertemplate=(
                    "birth = %{x:.4f}<br>"
                    "death = %{y:.4f}<br>"
                    "persistence = %{customdata[0]:.4f}<br>"
                    "tau = %{customdata[1]}<br>"
                    "dimension = %{customdata[2]}<extra></extra>"
                )
            )
        )

    # Диагональ birth = death.
    # Точки около диагонали обычно считаются менее устойчивыми.
    fig.add_trace(
        go.Scatter(
            x=[axis_min, axis_max],
            y=[axis_min, axis_max],
            mode="lines",
            name="birth = death",
            line=dict(dash="dash")
        )
    )

    tau_value = int(one["tau"].iloc[0])
    dimension_value = int(one["dimension"].iloc[0])

    infinite_count = one["is_infinite"].sum() if "is_infinite" in one.columns else 0

    fig.update_layout(
        title=(
            f"Диаграмма персистентности для ряда: {column_name}<br>"
            f"tau = {tau_value}, dimension = {dimension_value}, "
            f"бесконечных интервалов: {infinite_count}"
        ),
        xaxis_title="Birth",
        yaxis_title="Death",
        width=850,
        height=700,
        legend_title="Гомологии"
    )

    fig.update_xaxes(range=[axis_min, axis_max])
    fig.update_yaxes(range=[axis_min, axis_max], scaleanchor="x", scaleratio=1)

    fig.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

columns = sorted(diagrams["column"].dropna().unique())

column_dropdown = widgets.Dropdown(
    options=columns,
    value=columns[0],
    description="Ряд:",
    layout=widgets.Layout(width="700px")
)

output = widgets.Output()


def on_column_change(change):
    if change["name"] == "value":
        with output:
            clear_output(wait=True)
            plot_persistence_diagram_for_column(change["new"])


column_dropdown.observe(on_column_change)

display(column_dropdown)

with output:
    plot_persistence_diagram_for_column(column_dropdown.value)

display(output)

Dropdown(description='Ряд:', layout=Layout(width='700px'), options=('Lab1_G1_N1', 'Lab1_G1_N2', 'Lab1_G1_N3', …

Output()